# PlanWise Medicare Part D — Intake Question Notebook

**Purpose:** Capture member information for the Michigan stand-alone Medicare Part D MVP and pass it to the plan-comparison engine and adaptive explanatory summary.

This version follows the provided information from the **Question Bank: CMS Data Support Review**. It also keeps the tiers and retains the question asking whether the member is a first-time Medicare Part D user.

**MVP scope:** Predefined Michigan synthetic profiles, standardized medications, predefined pharmacy locations, and profiles without Extra Help.

## Design decisions from the CMS review

- ZIP code can help locate pharmacies, but the reviewed geographic data do not provide a ZIP-to-PDP-region crosswalk. ZIP code should not be used to filter the statewide Michigan PDP set.
- Ask for a **specific pharmacy location**, not just a chain name. A pharmacy directory or predefined synthetic locations are needed.
- For each medication, collect strength, dosage form, **quantity per fill**, and **days supplied**. Refill frequency alone is insufficient for the cost calculation.
- Keep the medication as entered. Generic substitution is deferred because the current data do not establish appropriate brand/generic substitution.
- Extra Help is a scope check only. The reviewed beneficiary-cost data are unsubsidized; the current cost model is not suitable for an unqualified personalized estimate when Extra Help is Yes or unknown.
- Mail-order cost comparisons are deferred. Average in-area retail pricing is not a verified mail-order price.
- Remove income-stability questions. The reviewed data do not support a plan-cost calculation from that answer.


## Connected CMS version
Taylor’s Tier 1–3 questions and record structure are retained. The catalog now includes 4,261 CMS-linked medication concepts, and the pharmacy list contains 20 verified Michigan locations for the demonstration. Original notebook: `reference/Taylor_Intake_v2_original.ipynb`.

Open from `planwise_data`, select the project Python environment, and Run All. The example runs offline using cached references. Other medications may need an explicit RxTerms lookup (`medication_details(rxcui, allow_network=True)`) for structured strength/form before selection. Pharmacy search is exact-ZIP within a limited directory, not a complete statewide or distance search. The result is data linkage, not a validated annual cost estimate.


## Tier 1 — Core profile

| Question | Response format | Use |
|---|---|---|
| Do you live in Michigan? | Yes / No | Confirms the prototype's geographic scope |
| What is your current ZIP code? | Five-digit ZIP | Helps locate pharmacies; does not filter statewide PDPs |
| Which medications do you take, including strength and form? | Search/select standardized medication | Matches the medication to CMS formulary and pricing identifiers |
| For each medication, how many units do you receive per fill, and how many days does that supply last? | Quantity + days supplied | Supports estimated prescription costs and applicable cost-sharing |
| Which pharmacy location do you use? | Specific location from predefined list | Identifies network participation and cost-sharing category |
| Do you currently receive Extra Help with prescription drug costs? | Yes / No / I don't know | Scope check; does not determine eligibility |
| Are you a first-time user of Medicare Part D? | Yes / No | Retained for explanatory context; not currently a cost-engine input |

**Medication catalog note:** Medication search, strengths, and dosage forms must be populated from the team's selected standardized source. The reviewed CMS files use identifiers such as RXCUI and NDC where available. Do not hard-code an invented universal medication/dosage list.


In [1]:
from pathlib import Path
import sys, json
ROOT=Path.cwd()
if not (ROOT/'data/planwise.duckdb').exists(): ROOT=ROOT/'planwise_data'
assert (ROOT/'data/planwise.duckdb').exists(), 'Open from planwise_data with its prepared database.'
sys.path.insert(0,str(ROOT/'scripts'))
from intake_data import (load_catalogs, select_medication, medication_details, ndc_choices,
                         valid_quantity_and_days, validate_selected_medications,
                         validate_pharmacy, plan_input_preview)
import re
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

YES_NO = ["Yes", "No"]
EXTRA_HELP_OPTIONS = ["Yes", "No", "I don't know"]

def valid_zip(zip_code):
    # Keep ZIP as a string to preserve leading zeroes.
    return bool(re.fullmatch(r"\d{5}", str(zip_code).strip()))

def valid_yes_no(answer):
    return answer in YES_NO

def valid_extra_help(answer):
    return answer in EXTRA_HELP_OPTIONS

# Connected source-backed catalogs; numerical validation is imported from intake_data.
MEDICATION_CATALOG, PHARMACY_LOCATIONS = load_catalogs()

def search_medications(query, catalog=None):
    """Simple search over catalog entries with a display_name field."""
    catalog = MEDICATION_CATALOG if catalog is None else catalog
    query = query.strip().lower()
    if not query:
        return []
    return [m for m in catalog if query in m.get("display_name", "").lower()]

def pharmacies_for_zip(zip_code, locations=None):
    """Placeholder filter; real 10–15 mile search needs geocoding + distance."""
    locations = PHARMACY_LOCATIONS if locations is None else locations
    return [p for p in locations if p.get("zip_code") == str(zip_code).strip()]

assert valid_zip("04817")
assert not valid_zip("4817")
assert valid_extra_help("I don't know")
assert valid_quantity_and_days(30, 30)
print("Tier 1 validation checks passed.")

print(f"Loaded {len(MEDICATION_CATALOG):,} medications and {len(PHARMACY_LOCATIONS)} pharmacy locations.")

Tier 1 validation checks passed.
Loaded 4,261 medications and 20 pharmacy locations.


### Medication entry fields

Each member can have multiple medication records.

| Field | Description |
|---|---|
| `medication_name` | Standardized medication display name |
| `rxcui` / `ndc` | Identifier, if available |
| `strength` | Selected strength |
| `dosage_form` | Tablet, capsule, liquid, etc. |
| `quantity_per_fill` | Number of units dispensed per fill |
| `days_supply` | Number of days covered by that fill |
| `assume_year_round_use` | Whether the synthetic profile assumes continued use throughout the year |

The exact dosage options should be linked to the selected medication in the catalog.


## Tier 2 — Comparison context

| Question | Response format | Use |
|---|---|---|
| Would you consider using another retail pharmacy? | Yes / No / Not sure | Determines whether the summary discusses alternatives |

**Deferred from the initial MVP:**
- Generic substitution: defer; compare the medication supplied in the profile unchanged.
- Whether a medication cannot be switched: unnecessary while the MVP does not substitute medications.
- Mail-order willingness and cost comparison: defer until mail-order pricing assumptions are resolved.

The current pricing file contains average in-area retail unit costs, not exact drug prices at each pharmacy. Do not promise precise savings from switching pharmacies.


In [2]:
ALTERNATIVE_PHARMACY_OPTIONS = ["Yes", "No", "Not sure"]

def valid_alternative_pharmacy(answer):
    return answer in ALTERNATIVE_PHARMACY_OPTIONS

assert valid_alternative_pharmacy("Not sure")
print("Tier 2 validation check passed.")


Tier 2 validation check passed.


## Tier 3 — Explanation preference

**Question:** What would you most like the comparison to explain?

Choose one:
1. Estimated annual total
2. Monthly premium and prescription payments
3. Why costs may vary during the year
4. Balanced overview

This changes the order and emphasis of the adaptive explanatory summary. All versions must still include essential costs, restrictions, and limitations.

The earlier “predictability versus lowest costs” question is replaced for this MVP. Premiums, deductibles, and cost-sharing can help explain tradeoffs, but do not by themselves establish that a plan's costs are more predictable. A predictability measure would need to be defined and tested.


In [3]:
EXPLANATION_FOCUS_OPTIONS = [
    "Estimated annual total",
    "Monthly premium and prescription payments",
    "Why costs may vary during the year",
    "Balanced overview",
]

def valid_explanation_focus(answer):
    return answer in EXPLANATION_FOCUS_OPTIONS

assert valid_explanation_focus("Balanced overview")
print("Tier 3 validation check passed.")


Tier 3 validation check passed.


## Create a structured intake record

The function validates the answers and stores them by tier. The Extra Help scope check prevents the current unsubsidized model from being presented as an unqualified personalized estimate for a member who answers Yes or I don't know.


In [4]:
def create_intake_record(
    lives_in_michigan,
    zip_code,
    medications,
    pharmacy_location,
    extra_help_status,
    first_time_part_d,
    consider_another_retail_pharmacy,
    explanation_focus,
):
    errors = []

    if not valid_yes_no(lives_in_michigan):
        errors.append("Michigan residence must be Yes or No.")
    elif lives_in_michigan != "Yes":
        errors.append("This prototype currently supports Michigan residents only.")

    if not valid_zip(zip_code):
        errors.append("ZIP code must contain exactly five digits.")

    if not isinstance(medications, list) or not medications:
        errors.append("Add at least one medication record.")
    else:
        for i, med in enumerate(medications, start=1):
            if not isinstance(med, dict):
                errors.append(f"Medication {i} must be a record.")
                continue
            for field in ["medication_name", "strength", "dosage_form"]:
                if not med.get(field):
                    errors.append(f"Medication {i} is missing {field}.")
            if not valid_quantity_and_days(
                med.get("quantity_per_fill"), med.get("days_supply")
            ):
                errors.append(
                    f"Medication {i} needs positive quantity_per_fill and days_supply."
                )

        errors.extend(validate_selected_medications(medications))

    errors.extend(validate_pharmacy(pharmacy_location))
    if not isinstance(pharmacy_location, dict) or not pharmacy_location.get("location_id"):
        errors.append("Choose a specific pharmacy location with a location_id.")

    if not valid_extra_help(extra_help_status):
        errors.append("Extra Help answer must be Yes, No, or I don't know.")

    if not valid_yes_no(first_time_part_d):
        errors.append("First-time Part D answer must be Yes or No.")

    if not valid_alternative_pharmacy(consider_another_retail_pharmacy):
        errors.append("Alternative pharmacy answer must be Yes, No, or Not sure.")

    if not valid_explanation_focus(explanation_focus):
        errors.append("Choose a valid explanation focus.")

    if errors:
        raise ValueError("\n".join(errors))

    cost_scope = (
        "Eligible for unsubsidized data checks; annual estimate not yet implemented"
        if extra_help_status == "No"
        else "Out of scope: do not present an unqualified personalized cost estimate"
    )

    return {
        "tier1": {
            "lives_in_michigan": lives_in_michigan,
            "zip_code": str(zip_code).strip(),
            "medications": medications,
            "pharmacy_location": pharmacy_location,
            "extra_help_status": extra_help_status,
            "first_time_part_d": first_time_part_d,
        },
        "tier2": {
            "consider_another_retail_pharmacy": consider_another_retail_pharmacy,
        },
        "tier3": {
            "explanation_focus": explanation_focus,
        },
        "scope_checks": {
            "cost_estimate_scope": cost_scope,
            "extra_help_eligibility_determined": False,
            "zip_used_to_filter_statewide_pdp_plans": False,
            "annual_cost_engine_validated": False,
        },
    }


## Example synthetic profile with real catalog selections

The member is fictional. The medication codes and pharmacy location are real, source-linked selections. This example explicitly selects two products and uses 30 units for 30 days; it does not infer a dose or recommend either drug. ZIP 48001 is used for the synthetic Michigan example. Taylor’s original 04817 remains only as a five-digit-format test, not a verified Michigan location.

In [5]:
# Inspect choices before selecting; incomplete names are not automatic matches.
display(pd.DataFrame(search_medications("atorvastatin 20 MG")))
display(pd.DataFrame(pharmacies_for_zip("48001")))
selected_pharmacy = next(p for p in PHARMACY_LOCATIONS if p['location_id']=='101649470568')
synthetic_member = create_intake_record(
    lives_in_michigan="Yes",
    zip_code="48001",
    medications=[
        select_medication("617310", quantity_per_fill=30, days_supply=30, ndc="60505257908"),
        select_medication("314076", quantity_per_fill=30, days_supply=30, ndc="68180098003"),
    ],
    pharmacy_location=selected_pharmacy,
    extra_help_status="No",
    first_time_part_d="Yes",
    consider_another_retail_pharmacy="Not sure",
    explanation_focus="Balanced overview",
)
synthetic_member

,rxcui,display_name,tty
0,597967,amlodipine 10 MG / atorvastatin 20 MG Oral Tablet,SCD
1,597974,amlodipine 2.5 MG / atorvastatin 20 MG Oral Ta...,SCD
2,597980,amlodipine 5 MG / atorvastatin 20 MG Oral Tablet,SCD
3,617310,atorvastatin 20 MG Oral Tablet,SCD


,location_id,pharmacy_number,npi,organization_name,address,city,state,zip_code,source_network_zip,location_name
0,101922895010,101922895010,1922895010,ALGONAC PHARMACY,423 MICHIGAN ST,ALGONAC,MI,48001,48001,"ALGONAC PHARMACY — 423 MICHIGAN ST, ALGONAC, M..."
1,101649470568,101649470568,1649470568,KROGER CO OF MICHIGAN,2600 POINTE TREMBLE RD,ALGONAC,MI,48001,48001,KROGER CO OF MICHIGAN — 2600 POINTE TREMBLE RD...
2,101992659809,101992659809,1992659809,PHARMWELL DRUGS LLC,2700 POINTE TREMBLE RD STE 400,ALGONAC,MI,48001,48001,PHARMWELL DRUGS LLC — 2700 POINTE TREMBLE RD S...


{'tier1': {'lives_in_michigan': 'Yes',
  'zip_code': '48001',
  'medications': [{'medication_name': 'atorvastatin 20 MG Oral Tablet',
    'rxcui': '617310',
    'ndc': '60505257908',
    'strength': '20 mg',
    'dosage_form': 'Oral Tablet',
    'quantity_per_fill': 30.0,
    'days_supply': 30,
    'assume_year_round_use': True},
   {'medication_name': 'lisinopril 10 MG Oral Tablet',
    'rxcui': '314076',
    'ndc': '68180098003',
    'strength': '10 mg',
    'dosage_form': 'Oral Tablet',
    'quantity_per_fill': 30.0,
    'days_supply': 30,
    'assume_year_round_use': True}],
  'pharmacy_location': {'location_id': '101649470568',
   'pharmacy_number': '101649470568',
   'npi': '1649470568',
   'organization_name': 'KROGER CO OF MICHIGAN',
   'address': '2600 POINTE TREMBLE RD',
   'city': 'ALGONAC',
   'state': 'MI',
   'zip_code': '48001',
   'source_network_zip': '48001',
   'location_name': 'KROGER CO OF MICHIGAN — 2600 POINTE TREMBLE RD, ALGONAC, MI 48001'},
  'extra_help_status

## Review captured answers


In [6]:
t1 = synthetic_member["tier1"]
t2 = synthetic_member["tier2"]
t3 = synthetic_member["tier3"]

overview = pd.DataFrame([
    ["Tier 1", "Live in Michigan?", t1["lives_in_michigan"]],
    ["Tier 1", "ZIP code", t1["zip_code"]],
    ["Tier 1", "Preferred pharmacy location", t1["pharmacy_location"]["location_name"]],
    ["Tier 1", "Currently receive Extra Help?", t1["extra_help_status"]],
    ["Tier 1", "First-time Part D user?", t1["first_time_part_d"]],
    ["Tier 2", "Consider another retail pharmacy?", t2["consider_another_retail_pharmacy"]],
    ["Tier 3", "Explanation focus", t3["explanation_focus"]],
], columns=["Tier", "Question", "Answer"])

display(overview)
display(pd.DataFrame(t1["medications"]))
display(pd.DataFrame([synthetic_member["scope_checks"]]))


,Tier,Question,Answer
0,Tier 1,Live in Michigan?,Yes
1,Tier 1,ZIP code,48001
2,Tier 1,Preferred pharmacy location,KROGER CO OF MICHIGAN — 2600 POINTE TREMBLE RD...
3,Tier 1,Currently receive Extra Help?,No
4,Tier 1,First-time Part D user?,Yes
5,Tier 2,Consider another retail pharmacy?,Not sure
6,Tier 3,Explanation focus,Balanced overview


,medication_name,rxcui,ndc,strength,dosage_form,quantity_per_fill,days_supply,assume_year_round_use
0,atorvastatin 20 MG Oral Tablet,617310,60505257908,20 mg,Oral Tablet,30.0,30,True
1,lisinopril 10 MG Oral Tablet,314076,68180098003,10 mg,Oral Tablet,30.0,30,True


,cost_estimate_scope,extra_help_eligibility_determined,zip_used_to_filter_statewide_pdp_plans,annual_cost_engine_validated
0,Eligible for unsubsidized data checks; annual ...,False,False,False


## Handoff to the adaptive explanatory summary

The intake record should go to the plan-comparison engine. The comparison engine calculates costs using CMS data and documented assumptions. The adaptive explanatory summary then changes **which results are explained first and how they are described**.

| Intake answer | Suggested explanation behavior |
|---|---|
| First-time Part D user = Yes | Explain Part D terminology and cost components more explicitly |
| Will consider another retail pharmacy = Yes | Discuss alternatives, with pricing limitations stated |
| Will consider another retail pharmacy = No | Focus on the selected location and its network/cost-sharing implications |
| Estimated annual total | Lead with annual estimate and its components |
| Monthly premium and prescription payments | Lead with premium and prescription payment breakdown |
| Why costs may vary | Explain modeled deductible, tiers, copays/coinsurance, and cost changes |
| Balanced overview | Present costs, coverage, and tradeoffs together |
| Extra Help = Yes or I don't know | State that the unsubsidized model may not apply; do not imply eligibility or present an unqualified personalized estimate |

**Summary guardrails**
- Once validated annual estimates are available, show them alongside monthly premium. The current integration returns data inputs only.
- Do not claim exact pharmacy-specific prices or precise savings from switching pharmacies.
- Do not model mail-order savings or generic substitutions in this MVP.
- Preserve unknown values and state what needs verification.
- Do not infer Extra Help eligibility.
- Keep essential costs, restrictions, assumptions, and limitations in every summary, regardless of the selected explanation focus.

**Source basis:** Team-provided *Question Bank: CMS Data Support Review*, including its “Proposed MVP Intake Questions” and “Initial evaluation scope.” The review references the [CMS 2026 quarterly file record layout](https://data.cms.gov/sites/default/files/2025-10/83da019f-fa24-483e-87de-cc089780a6a5/SPUFRecordLayout-2026.pdf).


## Connected plan data

Each selected medication is checked against every Michigan plan and the selected pharmacy. Missing coverage/supply/network records remain explicit. Prescription quantities are retained as inputs but are not turned into annual costs here. All restriction flags still require interpretation. `joined_not_annual_estimate` means the input join succeeded, not that coverage conditions are satisfied or that a recommendation has been made.

In [7]:
linked_inputs = plan_input_preview(synthetic_member)
assert len(linked_inputs)==24  # two medications across twelve plans
display(pd.DataFrame(linked_inputs))
print(pd.Series([r['data_status'] for r in linked_inputs]).value_counts().to_string())
(ROOT/'evidence/integrated_intake_example.json').write_text(json.dumps(synthetic_member,indent=2))
(ROOT/'evidence/integrated_intake_plan_inputs.json').write_text(json.dumps(linked_inputs,indent=2,default=str))

,contract_id,plan_id,plan_name,premium,deductible,rxcui,ndc,drug_name,tier,days_supply,unit_cost,cost_type_pref,cost_amt_pref,cost_type_nonpref,cost_amt_nonpref,prior_authorization_yn,step_therapy_yn,quantity_limit_yn,pharmacy_number,preferred_status_retail,data_status,requested_medication,quantity_per_fill
0,S4802,084,Wellcare Classic (PDP),0.000000,615.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.147600,1,0.000000,1,10.000000,N,N,Y,101649470568,Y,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
1,S4802,148,Wellcare Value Script (PDP),0.000000,615.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.109600,1,0.000000,1,15.000000,N,N,Y,101649470568,Y,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
2,S5584,001,Prescription Blue Select (PDP),78.400000,615.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.087600,1,1.000000,1,6.000000,N,N,Y,101649470568,Y,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
3,S5584,002,Prescription Blue Premium (PDP),106.700000,0.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.087600,1,1.000000,1,6.000000,N,N,Y,101649470568,Y,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
4,S5601,026,SilverScript Choice (PDP),78.300000,615.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.077300,0,0.000000,1,0.000000,N,N,Y,101649470568,N,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
5,S5617,221,HealthSpring Assurance Rx (PDP),0.000000,615.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.045300,2,0.050000,2,0.100000,N,N,Y,101649470568,N,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
6,S5617,363,HealthSpring Extra Rx (PDP),50.400000,615.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.079500,1,0.000000,1,6.000000,N,N,Y,101649470568,N,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
7,S5884,136,Humana Basic Rx Plan (PDP),0.000000,615.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,2,30,0.092800,0,0.000000,1,1.000000,N,N,N,101649470568,N,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
8,S5884,159,Humana Premier Rx Plan (PDP),98.100000,0.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.092800,1,0.000000,1,5.000000,N,N,N,101649470568,N,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0
9,S5884,192,Humana Value Rx Plan (PDP),0.000000,601.000000,617310,60505257908,atorvastatin 20 MG Oral Tablet,1,30,0.095700,1,0.000000,1,1.000000,N,N,N,101649470568,N,joined_not_annual_estimate,atorvastatin 20 MG Oral Tablet,30.0


joined_not_annual_estimate    24


18454

## Integration validation

These checks reject identifiers inconsistent with their labels, mismatched drug/NDC selections, invalid numerical entries, unsupported days supplied, and unknown pharmacy locations. Extra Help Yes/unknown blocks the data handoff for cost use. The existing explanation preferences and first-time-user answer remain in the tiered record.

In [8]:
from copy import deepcopy

def arguments(member):
    return {**deepcopy(member['tier1']), **deepcopy(member['tier2']), **deepcopy(member['tier3'])}

checks=[]
def rejected(label, change):
    args=arguments(synthetic_member)
    change(args)
    try:
        create_intake_record(**args)
    except ValueError:
        checks.append({'test':label,'result':'PASS'});return
    raise AssertionError(label+' was incorrectly accepted')

rejected('Unknown pharmacy',lambda a:a['pharmacy_location'].update(location_id='invented'))
rejected('Mismatched drug NDC',lambda a:a['medications'][0].update(ndc='68180098003'))
rejected('Incorrect strength',lambda a:a['medications'][0].update(strength='999 mg'))
rejected('Missing drug identifiers',lambda a:a['medications'][0].update(rxcui=None,ndc=None))
rejected('Nonfinite quantity',lambda a:a['medications'][0].update(quantity_per_fill=float('inf')))
rejected('Fractional days',lambda a:a['medications'][0].update(days_supply=30.5))
rejected('Unsupported days',lambda a:a['medications'][0].update(days_supply=45))
rejected('Non-Michigan residence',lambda a:a.update(lives_in_michigan='No'))
rejected('Invalid explanation option',lambda a:a.update(explanation_focus='invented'))
for answer in ['Yes', "I don't know"]:
    args=arguments(synthetic_member);args['extra_help_status']=answer
    record=create_intake_record(**args)
    assert plan_input_preview(record)==[]
    checks.append({'test':'Extra Help '+answer+' blocks cost handoff','result':'PASS'})
assert synthetic_member['tier1']['first_time_part_d']=='Yes'
assert synthetic_member['tier2']['consider_another_retail_pharmacy']=='Not sure'
assert synthetic_member['tier3']['explanation_focus']=='Balanced overview'
checks.append({'test':'All three tiers preserved','result':'PASS'})
display(pd.DataFrame(checks))
(ROOT/'evidence/intake_integration_checks.json').write_text(json.dumps(checks,indent=2))

,test,result
0,Unknown pharmacy,PASS
1,Mismatched drug NDC,PASS
2,Incorrect strength,PASS
3,Missing drug identifiers,PASS
4,Nonfinite quantity,PASS
5,Fractional days,PASS
6,Unsupported days,PASS
7,Non-Michigan residence,PASS
8,Invalid explanation option,PASS
9,Extra Help Yes blocks cost handoff,PASS


830